# Experiment 3: Single-Sensor Consistency (Duplicate and Across-Setup Comparisons)

## Overview
Evaluate the consistency and reusability of individual sensors when repeatedly measuring samples with the same concentration (100 CFU/mL, Salmonella Enteritidis/SE, turkey rinsate). We analyze repeated measures from the same sensor to quantify signal stability and degradation over time.

## Key Concepts
- **Duplicate measurements**: Adjacent test pairs per sensor (e.g., `Test1`–`Test2`, `Test3`–`Test4`) measure the same solution/setup. These reflect pure repeatability under identical conditions.
- **Across-setup measurements**: Non-adjacent test comparisons (e.g., `Test1` vs `Test5`) often involve a re-setup or slight differences. These reflect reproducibility across setups.

## Objectives (Protocol Requirements)
1. ✅ Load and organize SERS data by `sensor_id → test_id → concentration` using the unified I/O utilities.
2. ✅ Detect adjacent duplicate pairs per sensor (same day/setup measurements).
3. ✅ Detect non-adjacent across-setup pairs per sensor (different days/setups).
4. ✅ For each pair type and each concentration, compute consistency metrics:
   - Absolute difference metrics: MSE, RMSE, MAE
   - Spectral-shape metrics: cosine similarity, spectral angle mapper (SAM)
   - Variance-based metrics: coefficient of variation (CV) for peak and AUC
5. ✅ Compare duplicate vs across-setup repeatability.
6. ✅ **Sensor degradation over time**: Plot how sensor consistency changes with repeated uses.
7. ✅ **Failure detection**: Identify when sensors fail/degrade based on consistency thresholds.
8. ✅ **Reusability assessment**: Determine number of uses before failure/degradation.
9. ✅ **Failure rate analysis**: Assess whether failure rate meets industry standards.

## Protocol Compliance
This notebook implements all requirements for Experiment 3:
- ✅ Signal repeatability assessment
- ✅ Sensor degradation over time analysis
- ✅ Failure detection and counting
- ✅ Reusability assessment (number of uses before failure)
- ✅ Failure rate visualization and summary

Notes:
- Style and structure intentionally mirror `00_signal_concentration_relationship.ipynb` for consistency.
- This notebook is fully standalone and self-contained.


In [ ]:
# Imports and Setup
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cosine
from numpy.linalg import norm
import warnings

from sensd_sers_analysis.data import load_dataset_by_serotypes
from sensd_sers_analysis.utils import natural_sort

warnings.filterwarnings("ignore")

# Plotting style (mirror 00 notebook)
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

print("✅ Imports and natural sorting functions completed successfully")

## 1. Data Loading and Inspection

We load SERS data for Salmonella Enteritidis (SE) and Mix from the curated dataset. The dataset follows the unified naming scheme and is loaded via `load_dataset_by_serotypes`. We then inspect structure and organize by `sensor_id → test_id → concentration`.


In [ ]:
# Load SERS Data (Experiment 3 focus)
# --------------------------------------------------------------------------------------------------

# Define dataset location (mirrors 00 notebook structure)
data_folder = "../example_data/SERS Data 8 (April 2025)/"
signals_folders = ["March testing-Dilutions"]  # curated subset for speed/cleanliness
serotypes = ["ST", "SE", "Mix"]

try:
    data_df = load_dataset_by_serotypes(data_folder, signals_folders, serotypes)
    print(f"✅ Successfully loaded {len(data_df)} data entries")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    data_df = pd.DataFrame()

# Basic inspection
if not data_df.empty:
    print("📊 Dataset Structure Analysis:")
    print("=" * 50)
    print(f"Number of data entries: {len(data_df)}")
    print(f"Raman shift points: {len(data_df['raman_shift'].iloc[0])}")
    print(
        f"Raman shift range: {data_df['raman_shift'].iloc[0][0]:.1f} - "
        f"{data_df['raman_shift'].iloc[0][-1]:.1f} cm⁻¹"
    )
    print(f"Columns: {list(data_df.columns)}")

    # Natural sort sensor and test IDs
    sensor_ids = natural_sort([str(s) for s in data_df["sensor_id"].unique().tolist()])
    print(f"Sensors: {sensor_ids}")

    # Create composite unique_id for debugging
    data_df["unique_id"] = (
        data_df["sensor_id"].astype(str)
        + "_"
        + data_df["test_id"].astype(str)
        + "_"
        + data_df["serotype"].astype(str)
        + "_"
        + data_df["signal_index"].astype(str)
    )
else:
    print("❌ No data loaded - cannot proceed")

## 2. Duplicate Pair Detection

For each `sensor_id`, we consider adjacent test pairs (natural order by `TestN`):
- (`Test1`, `Test2`), (`Test3`, `Test4`), ...
These are treated as duplicate measurement pairs under identical conditions.

We will also generate non-adjacent pairs for across-setup comparisons (e.g., `Test1` vs `Test5`).


In [ ]:
def extract_test_number(test_id: str) -> int:
    """Extract numeric part from Test ID like 'Test12' -> 12."""
    import re

    m = re.search(r"(\d+)", str(test_id))
    return int(m.group(1)) if m else -1


def _normalized_concentrations(series: pd.Series) -> tuple:
    """Return a sorted tuple of unique concentrations rounded to 8 decimals."""
    if series.empty:
        return tuple()
    vals = np.asarray(series.unique(), dtype=float)
    vals = np.round(vals.astype(float), 8)
    vals.sort()
    return tuple(vals.tolist())


def detect_duplicate_and_across_pairs(
    df: pd.DataFrame, min_gap: int = 2
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Detect adjacent duplicate pairs and across-setup pairs per sensor.

    Duplicate pairs are STRICTLY odd-even consecutive tests per sensor:
    (Test1, Test2), (Test3, Test4), ... A duplicate pair is accepted only if the
    two tests have exactly the same set of concentration values (after rounding).

    Across-setup pairs are non-adjacent test comparisons with gap >= min_gap:
    (Test1, Test5), (Test1, Test7), etc. These reflect reproducibility across setups.

    Args:
        df: DataFrame with sensor_id, test_id, concentration columns
        min_gap: Minimum test number gap for across-setup pairs (default: 2)

    Returns:
        Tuple of (duplicate_pairs_df, across_setup_pairs_df)
    """
    records_dup: list[dict] = []
    records_across: list[dict] = []

    for sensor_id, g in df.groupby("sensor_id"):
        # Map test numbers to test IDs
        test_ids = g["test_id"].astype(str).unique().tolist()
        test_num_to_id = {extract_test_number(tid): tid for tid in test_ids}
        present_nums = sorted([n for n in test_num_to_id.keys() if n >= 0])

        # Strict odd-even consecutive pairs: n (odd) and n+1 (even) must both exist
        for n in present_nums:
            if n % 2 == 1 and (n + 1) in test_num_to_id:
                a = test_num_to_id[n]
                b = test_num_to_id[n + 1]

                # Robust identical concentration-set check (order/float safe)
                conc_a_tuple = _normalized_concentrations(g[g["test_id"] == a]["concentration"])
                conc_b_tuple = _normalized_concentrations(g[g["test_id"] == b]["concentration"])
                if conc_a_tuple and conc_a_tuple == conc_b_tuple:
                    records_dup.append(
                        {
                            "sensor_id": sensor_id,
                            "test_id_a": a,
                            "test_id_b": b,
                            "is_duplicate_pair": True,
                        }
                    )

        # Across-setup pairs: non-adjacent tests with gap >= min_gap
        for i, n_a in enumerate(present_nums):
            for n_b in present_nums[i + 1 :]:
                gap = n_b - n_a
                if gap >= min_gap:
                    a = test_num_to_id[n_a]
                    b = test_num_to_id[n_b]

                    # Check if they share at least one common concentration
                    conc_a_set = set(
                        _normalized_concentrations(g[g["test_id"] == a]["concentration"])
                    )
                    conc_b_set = set(
                        _normalized_concentrations(g[g["test_id"] == b]["concentration"])
                    )
                    if conc_a_set and conc_b_set and conc_a_set.intersection(conc_b_set):
                        records_across.append(
                            {
                                "sensor_id": sensor_id,
                                "test_id_a": a,
                                "test_id_b": b,
                                "test_gap": gap,
                                "is_across_setup": True,
                            }
                        )

    # Natural sort sensor_id and sort tests numerically for stable display/order
    def sort_pairs(df_pairs: pd.DataFrame) -> pd.DataFrame:
        if df_pairs.empty:
            return df_pairs
        sensor_order = natural_sort(df_pairs["sensor_id"].astype(str).unique().tolist())
        sensor_rank = {sid: i for i, sid in enumerate(sensor_order)}
        df_pairs = (
            df_pairs.assign(
                _sensor_rank=df_pairs["sensor_id"].map(sensor_rank),
                _a_num=df_pairs["test_id_a"].map(extract_test_number),
                _b_num=df_pairs["test_id_b"].map(extract_test_number),
            )
            .sort_values(["_sensor_rank", "_a_num", "_b_num"])
            .drop(columns=["_sensor_rank", "_a_num", "_b_num"])
            .reset_index(drop=True)
        )
        return df_pairs

    dup_df = sort_pairs(pd.DataFrame(records_dup))
    across_df = sort_pairs(pd.DataFrame(records_across))

    return dup_df, across_df


if not data_df.empty:
    duplicate_pairs_df, across_pairs_df = detect_duplicate_and_across_pairs(data_df, min_gap=2)
    print(f"✅ Duplicate pairs (after concentration check): {len(duplicate_pairs_df)}")
    print(f"✅ Across-setup pairs (gap >= 2): {len(across_pairs_df)}")

    if not duplicate_pairs_df.empty:
        print("\nSample duplicate pairs:")
        display(duplicate_pairs_df.head(10))

    if not across_pairs_df.empty:
        print("\nSample across-setup pairs:")
        display(across_pairs_df.head(10))
    else:
        print("\n⚠️ No across-setup pairs detected. This could mean:")
        print("   - All test pairs are adjacent (Test1-Test2, Test3-Test4, etc.)")
        print("   - Need tests with gap >= 2 (e.g., Test1-Test5, Test1-Test7)")
        print("   - Check if sensors have enough tests to form across-setup pairs")
else:
    duplicate_pairs_df = pd.DataFrame()
    across_pairs_df = pd.DataFrame()
    print("⚠️ No data available for pair detection")

## 3. Within-Duplicate Consistency Metrics

For each duplicate pair (`test_id_a`, `test_id_b`) within a `sensor_id` and for each concentration value present in both tests, compute per-signal spectral similarity metrics:
- Absolute difference: MSE, RMSE, MAE
- Spectral shape: Cosine similarity, Spectral Angle Mapper (SAM)
- Variance-based: Coefficient of Variation (CV) of max peak and AUC across the two signals

Notes:
- We compare signals at matching `concentration` values only.
- If a concentration appears multiple times (should not in current format), we align by sorted order.


In [ ]:
from scipy import integrate


def compute_signal_metrics_for_pair(
    sig_a: np.ndarray, sig_b: np.ndarray, raman_shift: np.ndarray
) -> dict:
    """Compute per-pair spectral similarity metrics.

    Returns a dictionary with MSE, RMSE, MAE, cosine_similarity, SAM (radians),
    and CV for max peak and AUC.
    """
    # Absolute difference metrics
    diff = sig_a - sig_b
    mse = float(np.mean(diff**2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(diff)))

    # Spectral shape metrics
    # Cosine distance from scipy is 1 - cosine_similarity
    cos_sim = float(1.0 - cosine(sig_a, sig_b))
    # Spectral Angle Mapper: arccos( dot(a,b) / (||a|| ||b||) )
    denom = norm(sig_a) * norm(sig_b)
    sam = (
        float(np.arccos(np.clip(np.dot(sig_a, sig_b) / denom, -1.0, 1.0))) if denom > 0 else np.nan
    )

    # Peak and AUC for each
    max_a = float(np.max(sig_a))
    max_b = float(np.max(sig_b))
    auc_a = float(integrate.trapezoid(sig_a, raman_shift))
    auc_b = float(integrate.trapezoid(sig_b, raman_shift))

    # Coefficient of Variation across duplicates: std/mean
    def coeff_var(vals: list[float]) -> float:
        vals_arr = np.asarray(vals, dtype=float)
        mu = np.mean(vals_arr)
        return float(np.std(vals_arr, ddof=1) / mu) if mu != 0 and len(vals_arr) >= 2 else np.nan

    cv_max = coeff_var([max_a, max_b])
    cv_auc = coeff_var([auc_a, auc_b])

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "cosine_similarity": cos_sim,
        "sam": sam,
        "cv_max": cv_max,
        "cv_auc": cv_auc,
        "max_a": max_a,
        "max_b": max_b,
        "auc_a": auc_a,
        "auc_b": auc_b,
    }


def compute_within_duplicate_metrics(
    df: pd.DataFrame, duplicate_pairs_df: pd.DataFrame
) -> pd.DataFrame:
    """Compute metrics for all duplicate pairs at matching concentrations.

    Returns a long-form DataFrame with identification and metrics per pair and concentration.
    """
    results: list[dict] = []

    # Group data for quick access
    for _, pair in duplicate_pairs_df.iterrows():
        sensor_id = pair["sensor_id"]
        test_a = pair["test_id_a"]
        test_b = pair["test_id_b"]

        g_sensor = df[df["sensor_id"] == sensor_id]
        g_a = g_sensor[g_sensor["test_id"] == test_a]
        g_b = g_sensor[g_sensor["test_id"] == test_b]

        if g_a.empty or g_b.empty:
            continue

        # Identify common concentrations
        conc_a = set(g_a["concentration"].unique().tolist())
        conc_b = set(g_b["concentration"].unique().tolist())
        common_conc = sorted(list(conc_a.intersection(conc_b)))

        for conc in common_conc:
            rows_a = g_a[g_a["concentration"] == conc].sort_values("signal_index")
            rows_b = g_b[g_b["concentration"] == conc].sort_values("signal_index")

            # Align by order; expect one per concentration
            if len(rows_a) == 0 or len(rows_b) == 0:
                continue

            row_a = rows_a.iloc[0]
            row_b = rows_b.iloc[0]

            sig_a = row_a["signal"]
            sig_b = row_b["signal"]
            raman_shift = row_a["raman_shift"]

            # Safety check for alignment
            if len(sig_a) != len(sig_b) or len(sig_a) != len(raman_shift):
                continue

            metrics = compute_signal_metrics_for_pair(sig_a, sig_b, raman_shift)
            results.append(
                {
                    "sensor_id": sensor_id,
                    "test_id_a": test_a,
                    "test_id_b": test_b,
                    "concentration": float(conc),
                    **metrics,
                }
            )

    return pd.DataFrame(results)


if not data_df.empty and not duplicate_pairs_df.empty:
    within_dup_metrics_df = compute_within_duplicate_metrics(data_df, duplicate_pairs_df)
    print(f"✅ Within-duplicate metrics computed: {len(within_dup_metrics_df)} rows")
    display(within_dup_metrics_df.head())
else:
    within_dup_metrics_df = pd.DataFrame()

In [ ]:
# Sensor-level global consistency indices (All-pairs vs Duplicate-only)
# --------------------------------------------------------------------------------------------------

from itertools import combinations


def pairwise_metrics_for_sensor(df: pd.DataFrame, sensor_id: str) -> pd.DataFrame:
    """Compute metrics for all same-concentration pairs across any tests within a sensor.

    For each concentration, form all combinations of tests and compute metrics using
    the first signal per test at that concentration.
    """
    g = df[df["sensor_id"] == sensor_id]
    results: list[dict] = []

    for conc, g_conc in g.groupby("concentration"):
        # test_id -> first row at this concentration
        reps = g_conc.sort_values(["test_id", "signal_index"]).groupby("test_id").head(1)
        test_ids = reps["test_id"].tolist()
        rows = list(reps.itertuples(index=False))
        # Pre-extract signal and raman_shift in same order
        signals = [r.signal for r in rows]
        raman_shifts = [r.raman_shift for r in rows]

        # Only compare if we have at least 2 tests
        if len(rows) < 2:
            continue

        # Verify same raman_shift within this concentration group
        ref = raman_shifts[0]
        ok = all(len(ref) == len(rs) and np.array_equal(ref, rs) for rs in raman_shifts[1:])
        if not ok:
            continue

        for i, j in combinations(range(len(rows)), 2):
            sig_a = signals[i]
            sig_b = signals[j]
            metrics = compute_signal_metrics_for_pair(sig_a, sig_b, ref)
            results.append(
                {
                    "sensor_id": sensor_id,
                    "test_id_a": test_ids[i],
                    "test_id_b": test_ids[j],
                    "concentration": float(conc),
                    **metrics,
                }
            )

    return pd.DataFrame(results)


def aggregate_consistency(metrics_df: pd.DataFrame, by: str = "sensor_id") -> pd.DataFrame:
    """Aggregate metrics to a single consistency index per group.

    We report means for error-based metrics (lower is better) and for similarity-based
    metrics (higher is better). The caller will label which set it is (all-pairs vs duplicates).
    """
    if metrics_df.empty:
        return pd.DataFrame()

    agg = (
        metrics_df.groupby(by)
        .agg(
            rmse_mean=("rmse", "mean"),
            mae_mean=("mae", "mean"),
            mse_mean=("mse", "mean"),
            cosine_similarity_mean=("cosine_similarity", "mean"),
            sam_mean=("sam", "mean"),
            cv_max_mean=("cv_max", "mean"),
            cv_auc_mean=("cv_auc", "mean"),
            n_pairs=("rmse", "size"),
        )
        .reset_index()
    )
    return agg


# Compute all-pairs metrics per sensor
if not data_df.empty:
    all_pairs_list: list[pd.DataFrame] = []
    for sid in natural_sort(data_df["sensor_id"].astype(str).unique().tolist()):
        all_pairs_list.append(pairwise_metrics_for_sensor(data_df, sid))
    all_pairs_metrics_df = (
        pd.concat([df for df in all_pairs_list if not df.empty], ignore_index=True)
        if all_pairs_list
        else pd.DataFrame()
    )
    print(
        f"✅ All-pairs metrics rows: {0 if all_pairs_metrics_df is None or all_pairs_metrics_df.empty else len(all_pairs_metrics_df)}"
    )
else:
    all_pairs_metrics_df = pd.DataFrame()

# Aggregate duplicate-only vs all-pairs at sensor-level
duplicate_sensor_consistency = (
    aggregate_consistency(within_dup_metrics_df, by="sensor_id")
    if "within_dup_metrics_df" in globals()
    else pd.DataFrame()
)
allpairs_sensor_consistency = aggregate_consistency(all_pairs_metrics_df, by="sensor_id")

# Join for comparison
sensor_consistency = duplicate_sensor_consistency.merge(
    allpairs_sensor_consistency,
    on="sensor_id",
    how="outer",
    suffixes=("_duplicate", "_allpairs"),
)

# Natural order for display
if not sensor_consistency.empty:
    sensor_consistency = (
        sensor_consistency.set_index("sensor_id")
        .loc[
            natural_sort(sensor_consistency["sensor_id"].astype(str).tolist())
            if "sensor_id" in sensor_consistency.columns
            else natural_sort(sensor_consistency.index.astype(str).tolist())
        ]
        .reset_index()
    )

print("\n📊 Sensor-level consistency summary (means across pairs):")
display(sensor_consistency.head(20))

In [ ]:
# Visualization: Duplicate-only vs All-pairs consistency per sensor
# --------------------------------------------------------------------------------------------------


def plot_sensor_consistency_comparison(
    sensor_consistency: pd.DataFrame, metric: str = "rmse_mean"
) -> None:
    if sensor_consistency.empty:
        print("❌ No data for sensor-level visualization")
        return

    dup_col = f"{metric}_duplicate"
    all_col = f"{metric}_allpairs"
    if dup_col not in sensor_consistency.columns or all_col not in sensor_consistency.columns:
        print(f"❌ Missing expected columns: {dup_col} and/or {all_col}")
        return

    # Prepare tidy format
    plot_df = sensor_consistency[["sensor_id", dup_col, all_col]].copy()
    plot_df = plot_df.rename(columns={dup_col: "duplicate", all_col: "allpairs"})
    plot_df = plot_df.melt(
        id_vars=["sensor_id"],
        value_vars=["duplicate", "allpairs"],
        var_name="pair_type",
        value_name=metric,
    )

    # Order sensors naturally
    sensors = natural_sort(plot_df["sensor_id"].astype(str).unique().tolist())
    plt.figure(figsize=(8, 4))
    sns.barplot(data=plot_df, x="sensor_id", y=metric, hue="pair_type", order=sensors)
    plt.xticks(rotation=45)
    plt.title(f"Sensor-level {metric} comparison: duplicate vs all-pairs")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


# Draw several metrics to highlight the story
if "sensor_consistency" in globals() and not sensor_consistency.empty:
    for metric in ["rmse_mean", "mae_mean", "cosine_similarity_mean"]:
        # Only plot if both duplicate and allpairs columns exist
        need_cols = [f"{metric}_duplicate", f"{metric}_allpairs"]
        if all(col in sensor_consistency.columns for col in need_cols):
            plot_sensor_consistency_comparison(sensor_consistency, metric)

## 4. Across-Day/Across-Setup Consistency Metrics

For each sensor, compute the same metrics for non-adjacent test pairs (gap ≥ 2). This approximates differences across re-setup or day-to-day variations.


In [ ]:
# Compute across-setup metrics using the same per-pair computation
# --------------------------------------------------------------------------------------------------


def compute_across_setup_metrics(df: pd.DataFrame, across_pairs_df: pd.DataFrame) -> pd.DataFrame:
    """Compute metrics for non-adjacent pairs at matching concentrations."""
    # Reuse the same logic as duplicates by calling the same function body
    # We'll copy the minimal alignment logic here for clarity
    results: list[dict] = []

    for _, pair in across_pairs_df.iterrows():
        sensor_id = pair["sensor_id"]
        test_a = pair["test_id_a"]
        test_b = pair["test_id_b"]

        g_sensor = df[df["sensor_id"] == sensor_id]
        g_a = g_sensor[g_sensor["test_id"] == test_a]
        g_b = g_sensor[g_sensor["test_id"] == test_b]
        if g_a.empty or g_b.empty:
            continue

        conc_a = set(g_a["concentration"].unique().tolist())
        conc_b = set(g_b["concentration"].unique().tolist())
        common_conc = sorted(list(conc_a.intersection(conc_b)))

        for conc in common_conc:
            rows_a = g_a[g_a["concentration"] == conc].sort_values("signal_index")
            rows_b = g_b[g_b["concentration"] == conc].sort_values("signal_index")
            if len(rows_a) == 0 or len(rows_b) == 0:
                continue

            row_a = rows_a.iloc[0]
            row_b = rows_b.iloc[0]

            sig_a = row_a["signal"]
            sig_b = row_b["signal"]
            raman_shift = row_a["raman_shift"]
            if len(sig_a) != len(sig_b) or len(sig_a) != len(raman_shift):
                continue

            metrics = compute_signal_metrics_for_pair(sig_a, sig_b, raman_shift)
            results.append(
                {
                    "sensor_id": sensor_id,
                    "test_id_a": test_a,
                    "test_id_b": test_b,
                    "concentration": float(conc),
                    **metrics,
                }
            )

    return pd.DataFrame(results)


if not data_df.empty and not across_pairs_df.empty:
    print(f"📊 Computing metrics for {len(across_pairs_df)} across-setup pairs...")
    across_setup_metrics_df = compute_across_setup_metrics(data_df, across_pairs_df)
    print(f"✅ Across-setup metrics computed: {len(across_setup_metrics_df)} rows")
    if not across_setup_metrics_df.empty:
        display(across_setup_metrics_df.head())
        print(f"📊 Across-setup metrics columns: {list(across_setup_metrics_df.columns)}")
    else:
        print("⚠️ Warning: Across-setup metrics DataFrame is empty!")
        print("   This might mean:")
        print("   - No matching concentrations found between test pairs")
        print("   - Signal alignment issues")
        print("   - Check across_pairs_df to see if pairs were detected correctly")
else:
    across_setup_metrics_df = pd.DataFrame()
    if data_df.empty:
        print("⚠️ Cannot compute across-setup metrics: data_df is empty")
    elif across_pairs_df.empty:
        print("⚠️ Cannot compute across-setup metrics: across_pairs_df is empty")
        print(
            "   This means no across-setup pairs were detected (all pairs might be adjacent duplicates)"
        )

## 5. Results Summary & Visualization

We summarize metrics per sensor, comparing within-duplicate vs across-setup distributions. Sensors are plotted in natural order (`modelC-1`, `modelC-2`, ...).


In [ ]:
def prepare_long_summary(within_df: pd.DataFrame, across_df: pd.DataFrame) -> pd.DataFrame:
    """Combine within-duplicate and across-setup metrics in long format.

    Handles cases where one or both DataFrames might be empty.
    Only includes columns that exist in both DataFrames (or in the non-empty one).
    """
    w = within_df.copy()
    a = across_df.copy()

    # Handle empty DataFrames
    if w.empty and a.empty:
        return pd.DataFrame()

    # If one is empty, use columns from the non-empty one
    if w.empty:
        print("⚠️ Warning: Duplicate metrics DataFrame is empty, only using across-setup data")
        a["pair_type"] = "across_setup"
        return a
    if a.empty:
        print("⚠️ Warning: Across-setup metrics DataFrame is empty, only using duplicate data")
        w["pair_type"] = "duplicate"
        return w

    # Both are non-empty: find common columns
    w_cols = set(w.columns)
    a_cols = set(a.columns)
    common_cols = list(w_cols.intersection(a_cols))

    # Required columns that must exist in both
    required_cols = ["sensor_id", "concentration"]
    missing_required = [col for col in required_cols if col not in common_cols]
    if missing_required:
        print(f"⚠️ Warning: Missing required columns in intersection: {missing_required}")
        print(f"   Duplicate columns: {list(w.columns)}")
        print(f"   Across-setup columns: {list(a.columns)}")
        return pd.DataFrame()

    # Desired metric columns (use only those that exist in both)
    desired_metrics = ["mse", "rmse", "mae", "cosine_similarity", "sam", "cv_max", "cv_auc"]
    available_metrics = [col for col in desired_metrics if col in common_cols]

    # Select only columns that exist in both DataFrames
    cols_to_use = required_cols + available_metrics

    w["pair_type"] = "duplicate"
    a["pair_type"] = "across_setup"

    # Only select columns that exist in both
    w_selected = w[cols_to_use + ["pair_type"]]
    a_selected = a[cols_to_use + ["pair_type"]]

    combined = pd.concat([w_selected, a_selected], ignore_index=True)

    # Natural order sensors
    if not combined.empty and "sensor_id" in combined.columns:
        combined["sensor_id"] = pd.Categorical(
            combined["sensor_id"],
            categories=natural_sort(combined["sensor_id"].astype(str).unique().tolist()),
            ordered=True,
        )

    return combined


def plot_metric_boxplots(summary_df: pd.DataFrame, metric: str) -> None:
    if summary_df.empty:
        print("❌ No summary data to plot")
        return
    plt.figure(figsize=(14, 6))
    sns.boxplot(data=summary_df, x="sensor_id", y=metric, hue="pair_type")
    plt.xticks(rotation=45)
    plt.title(f"{metric.replace('_', ' ').upper()} by Sensor: Duplicate vs Across-Setup")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_sensor_level_lines(summary_df: pd.DataFrame, metric: str) -> None:
    if summary_df.empty:
        return
    # Plot median metric per sensor for each pair type
    med = (
        summary_df.groupby(["sensor_id", "pair_type"], observed=True)[metric].median().reset_index()
    )
    sensors = med["sensor_id"].unique().tolist()
    plt.figure(figsize=(14, 5))
    for pair_type, df_t in med.groupby("pair_type"):
        plt.plot(sensors, df_t.set_index("sensor_id")[metric], marker="o", label=pair_type)
    plt.xticks(rotation=45)
    plt.ylabel(metric)
    plt.title(f"Median {metric} per sensor")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


if not within_dup_metrics_df.empty or not across_setup_metrics_df.empty:
    # Diagnostic: Check what columns exist
    if not within_dup_metrics_df.empty:
        print(f"📊 Duplicate metrics columns: {list(within_dup_metrics_df.columns)}")
    if not across_setup_metrics_df.empty:
        print(f"📊 Across-setup metrics columns: {list(across_setup_metrics_df.columns)}")

    summary_df = prepare_long_summary(within_dup_metrics_df, across_setup_metrics_df)

    if summary_df.empty:
        print("⚠️ Summary DataFrame is empty. Check if both DataFrames have matching columns.")
    else:
        print(f"✅ Summary rows: {len(summary_df)}")
        print(
            f"📊 Available metrics in summary: {[col for col in summary_df.columns if col not in ['sensor_id', 'concentration', 'pair_type']]}"
        )

        # Only plot metrics that exist in the summary
        available_metrics = [
            col
            for col in ["rmse", "mae", "cosine_similarity", "sam", "cv_max", "cv_auc"]
            if col in summary_df.columns
        ]

        if available_metrics:
            for metric in available_metrics:
                plot_metric_boxplots(summary_df, metric)
                plot_sensor_level_lines(summary_df, metric)
        else:
            print("⚠️ No metrics available for plotting")
else:
    summary_df = pd.DataFrame()
    print("⚠️ No metrics data available (both DataFrames are empty)")

## 6. Sensor Degradation Over Time Analysis (Protocol Requirement)

**Protocol Objective**: Determine the number of repeated uses a sensor can withstand before signal degradation or failure occurs.

We analyze how sensor consistency changes over repeated uses by plotting metrics vs test number for each sensor. This allows us to:
- Identify when sensors start degrading
- Determine the number of uses before degradation
- Assess sensor reusability


In [ ]:
# Sensor Degradation Over Time Analysis
# --------------------------------------------------------------------------------------------------
# Note: extract_test_number is already defined earlier in the notebook


def compute_sensor_trajectories(
    metrics_df: pd.DataFrame, reference_test: str = "Test1"
) -> pd.DataFrame:
    """Compute sensor consistency metrics over time (test number).

    For each sensor, compute metrics comparing each test to a reference test (typically Test1).
    This shows how sensor performance degrades over repeated uses.

    Args:
        metrics_df: DataFrame with metrics for pairs (can be duplicate or across-setup)
        reference_test: Test ID to use as reference (default: "Test1")

    Returns:
        DataFrame with columns: sensor_id, test_number, metric values
    """
    if metrics_df.empty:
        return pd.DataFrame()

    # For each sensor, find pairs that include the reference test
    trajectories = []

    for sensor_id in metrics_df["sensor_id"].unique():
        sensor_data = metrics_df[metrics_df["sensor_id"] == sensor_id].copy()

        # Find pairs that include the reference test
        ref_pairs = sensor_data[
            (sensor_data["test_id_a"] == reference_test)
            | (sensor_data["test_id_b"] == reference_test)
        ].copy()

        if ref_pairs.empty:
            continue

        # Extract test number and metric values
        for _, row in ref_pairs.iterrows():
            test_a = row["test_id_a"]
            test_b = row["test_id_b"]

            # Determine which test is the comparison test (not reference)
            if test_a == reference_test:
                comparison_test = test_b
            else:
                comparison_test = test_a

            test_num = extract_test_number(comparison_test)
            if test_num <= 0:
                continue

            # Extract metrics
            traj_row = {
                "sensor_id": sensor_id,
                "test_number": test_num,
                "test_id": comparison_test,
            }

            # Add all metric columns
            metric_cols = ["rmse", "mae", "mse", "cosine_similarity", "sam", "cv_max", "cv_auc"]
            for col in metric_cols:
                if col in row.index:
                    traj_row[col] = row[col]

            trajectories.append(traj_row)

    return pd.DataFrame(trajectories)


def plot_sensor_degradation_trajectories(
    trajectories_df: pd.DataFrame, metric: str = "rmse"
) -> None:
    """Plot sensor degradation trajectories over test number.

    Shows how each sensor's consistency changes over repeated uses.
    """
    if trajectories_df.empty or metric not in trajectories_df.columns:
        print(f"⚠️ No trajectory data available for metric: {metric}")
        return

    # Group by sensor
    sensors = natural_sort(trajectories_df["sensor_id"].astype(str).unique().tolist())

    fig, ax = plt.subplots(figsize=(14, 8))

    for sensor_id in sensors:
        sensor_traj = trajectories_df[trajectories_df["sensor_id"] == sensor_id].sort_values(
            "test_number"
        )
        if sensor_traj.empty:
            continue

        ax.plot(
            sensor_traj["test_number"],
            sensor_traj[metric],
            marker="o",
            label=sensor_id,
            alpha=0.7,
            linewidth=2,
            markersize=6,
        )

    ax.set_xlabel("Test Number (Use Number)", fontsize=12)
    ax.set_ylabel(metric.replace("_", " ").upper(), fontsize=12)
    ax.set_title(
        f"Sensor Degradation Over Time: {metric.replace('_', ' ').upper()}\n(Each line shows how sensor consistency changes with repeated use)",
        fontsize=14,
    )
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Compute trajectories using duplicate pairs (comparing to Test1)
if not within_dup_metrics_df.empty:
    # For degradation analysis, we want to compare each test to Test1
    # We'll use across-setup pairs that include Test1, or create them if needed

    # First, create Test1 reference comparisons if they don't exist in across-setup pairs
    test1_trajectories = []

    for sensor_id in data_df["sensor_id"].unique():
        sensor_data = data_df[data_df["sensor_id"] == sensor_id]
        test1_data = sensor_data[sensor_data["test_id"] == "Test1"]

        if test1_data.empty:
            continue

        # Get all other tests for this sensor
        other_tests = sensor_data[sensor_data["test_id"] != "Test1"]["test_id"].unique()

        for test_id in other_tests:
            test_num = extract_test_number(test_id)
            if test_num <= 1:
                continue

            # Get data for Test1 and this test
            test_data = sensor_data[sensor_data["test_id"] == test_id]

            # Find common concentrations
            conc_test1 = set(test1_data["concentration"].unique())
            conc_test = set(test_data["concentration"].unique())
            common_conc = sorted(list(conc_test1.intersection(conc_test)))

            for conc in common_conc:
                row1 = test1_data[test1_data["concentration"] == conc].iloc[0]
                row_test = test_data[test_data["concentration"] == conc].iloc[0]

                sig1 = np.asarray(row1["signal"], dtype=float)
                sig_test = np.asarray(row_test["signal"], dtype=float)
                raman_shift = np.asarray(row1["raman_shift"], dtype=float)

                if len(sig1) != len(sig_test) or len(sig1) != len(raman_shift):
                    continue

                metrics = compute_signal_metrics_for_pair(sig1, sig_test, raman_shift)

                traj_row = {
                    "sensor_id": sensor_id,
                    "test_number": test_num,
                    "test_id": test_id,
                    "concentration": float(conc),
                    **metrics,
                }
                test1_trajectories.append(traj_row)

    degradation_trajectories_df = pd.DataFrame(test1_trajectories)

    if not degradation_trajectories_df.empty:
        print(
            f"✅ Computed degradation trajectories: {len(degradation_trajectories_df)} data points"
        )
        print(f"   Sensors: {len(degradation_trajectories_df['sensor_id'].unique())}")
        print(
            f"   Test range: {degradation_trajectories_df['test_number'].min()} - {degradation_trajectories_df['test_number'].max()}"
        )

        # Plot key metrics
        for metric in ["rmse", "mae", "cosine_similarity"]:
            if metric in degradation_trajectories_df.columns:
                plot_sensor_degradation_trajectories(degradation_trajectories_df, metric)
    else:
        print("⚠️ Could not compute degradation trajectories")
        degradation_trajectories_df = pd.DataFrame()
else:
    degradation_trajectories_df = pd.DataFrame()
    print("⚠️ No duplicate metrics available for degradation analysis")

## 7. Failure Detection and Reusability Assessment (Protocol Requirement)

**Protocol Objectives**:
- Determine the number of repeated uses a sensor can withstand before signal degradation or failure occurs
- Assess whether the observed failure rate is acceptable by industry standards

We define sensor failure/degradation based on consistency metric thresholds and detect when sensors exceed these thresholds.


In [ ]:
# Failure Detection and Reusability Assessment
# --------------------------------------------------------------------------------------------------

# Define degradation/failure thresholds
# These can be adjusted based on industry standards or experimental observations
DEGRADATION_THRESHOLDS = {
    "rmse": None,  # Will be set based on baseline (e.g., 2× baseline)
    "mae": None,
    "cosine_similarity": 0.85,  # Below this = degraded (shape similarity drops)
    "sam": None,  # Will be set based on baseline
    "cv_max": 0.20,  # Above 20% CV = high variability
    "cv_auc": 0.20,
}

# Baseline: Use first few tests (Test1-Test3) as baseline for threshold calculation
BASELINE_TESTS = [1, 2, 3]  # Test numbers to use as baseline


def compute_baseline_metrics(trajectories_df: pd.DataFrame) -> dict:
    """Compute baseline metrics from early tests to define degradation thresholds."""
    if trajectories_df.empty:
        return {}

    baseline_data = trajectories_df[trajectories_df["test_number"].isin(BASELINE_TESTS)]

    if baseline_data.empty:
        return {}

    baselines = {}
    for metric in ["rmse", "mae", "cosine_similarity", "sam", "cv_max", "cv_auc"]:
        if metric in baseline_data.columns:
            values = baseline_data[metric].dropna()
            if len(values) > 0:
                baselines[metric] = {
                    "mean": values.mean(),
                    "std": values.std(),
                    "median": values.median(),
                    "p95": values.quantile(0.95) if len(values) > 1 else values.iloc[0],
                }

    return baselines


def detect_sensor_failures(
    trajectories_df: pd.DataFrame, baselines: dict, thresholds: dict
) -> pd.DataFrame:
    """Detect when sensors fail/degrade based on thresholds.

    Returns DataFrame with failure information per sensor.
    """
    if trajectories_df.empty:
        return pd.DataFrame()

    # Set dynamic thresholds based on baselines
    dynamic_thresholds = thresholds.copy()
    for metric in ["rmse", "mae", "sam"]:
        if metric in baselines and dynamic_thresholds.get(metric) is None:
            # Use 2× baseline mean + 2× std as threshold
            baseline_mean = baselines[metric]["mean"]
            baseline_std = baselines[metric]["std"]
            dynamic_thresholds[metric] = baseline_mean + 2 * baseline_std

    failures = []

    for sensor_id in trajectories_df["sensor_id"].unique():
        sensor_traj = trajectories_df[trajectories_df["sensor_id"] == sensor_id].sort_values(
            "test_number"
        )

        if sensor_traj.empty:
            continue

        # Check each test for failure
        failed = False
        failure_test_num = None
        failure_reason = None

        for _, row in sensor_traj.iterrows():
            test_num = row["test_number"]

            # Check each metric against threshold
            for metric, threshold in dynamic_thresholds.items():
                if threshold is None or metric not in row.index:
                    continue

                value = row[metric]
                if pd.isna(value):
                    continue

                # For error metrics (rmse, mae, sam, cv), higher is worse
                # For similarity metrics (cosine_similarity), lower is worse
                is_error_metric = metric in ["rmse", "mae", "sam", "cv_max", "cv_auc"]

                if is_error_metric and value > threshold:
                    failed = True
                    failure_test_num = test_num
                    failure_reason = f"{metric} > {threshold:.3f} (value: {value:.3f})"
                    break
                elif not is_error_metric and metric == "cosine_similarity" and value < threshold:
                    failed = True
                    failure_test_num = test_num
                    failure_reason = f"{metric} < {threshold:.3f} (value: {value:.3f})"
                    break

            if failed:
                break

        # Get final metrics for this sensor
        final_test_num = sensor_traj["test_number"].max()
        final_metrics = (
            sensor_traj[sensor_traj["test_number"] == final_test_num].iloc[0]
            if not sensor_traj.empty
            else None
        )

        failures.append(
            {
                "sensor_id": sensor_id,
                "failed": failed,
                "failure_test_number": failure_test_num,
                "uses_before_failure": failure_test_num if failed else final_test_num,
                "total_uses": final_test_num,
                "failure_reason": failure_reason if failed else "No failure detected",
                "final_rmse": final_metrics["rmse"]
                if final_metrics is not None and "rmse" in final_metrics.index
                else None,
                "final_cosine_similarity": final_metrics["cosine_similarity"]
                if final_metrics is not None and "cosine_similarity" in final_metrics.index
                else None,
            }
        )

    return pd.DataFrame(failures)


# Compute baselines and detect failures
if not degradation_trajectories_df.empty:
    baselines = compute_baseline_metrics(degradation_trajectories_df)

    if baselines:
        print("\n📊 Baseline Metrics (from Tests 1-3):")
        print("=" * 60)
        for metric, stats in baselines.items():
            print(f"{metric.upper()}:")
            print(f"  Mean: {stats['mean']:.4f} ± {stats['std']:.4f}")
            print(f"  Median: {stats['median']:.4f}")
            print(f"  95th percentile: {stats['p95']:.4f}")

        # Detect failures
        failures_df = detect_sensor_failures(
            degradation_trajectories_df, baselines, DEGRADATION_THRESHOLDS
        )

        if not failures_df.empty:
            print(f"\n✅ Failure detection completed for {len(failures_df)} sensors")

            # Summary statistics
            n_failed = failures_df["failed"].sum()
            n_total = len(failures_df)
            failure_rate = (n_failed / n_total * 100) if n_total > 0 else 0

            print("\n📊 Failure Summary:")
            print("=" * 60)
            print(f"Total sensors: {n_total}")
            print(f"Sensors failed: {n_failed} ({failure_rate:.1f}%)")
            print(f"Sensors operational: {n_total - n_failed} ({100 - failure_rate:.1f}%)")

            if n_failed > 0:
                failed_sensors = failures_df[failures_df["failed"]]
                print("\nUses before failure (failed sensors):")
                print(f"  Mean: {failed_sensors['uses_before_failure'].mean():.1f}")
                print(f"  Median: {failed_sensors['uses_before_failure'].median():.1f}")
                print(f"  Min: {failed_sensors['uses_before_failure'].min():.0f}")
                print(f"  Max: {failed_sensors['uses_before_failure'].max():.0f}")

            operational_sensors = failures_df[~failures_df["failed"]]
            if len(operational_sensors) > 0:
                print("\nTotal uses (operational sensors):")
                print(f"  Mean: {operational_sensors['total_uses'].mean():.1f}")
                print(f"  Median: {operational_sensors['total_uses'].median():.1f}")
                print(f"  Min: {operational_sensors['total_uses'].min():.0f}")
                print(f"  Max: {operational_sensors['total_uses'].max():.0f}")

            print("\n📋 Per-Sensor Failure Status:")
            display(failures_df)
        else:
            failures_df = pd.DataFrame()
            print("⚠️ Could not detect failures")
    else:
        failures_df = pd.DataFrame()
        print("⚠️ Could not compute baselines")
else:
    failures_df = pd.DataFrame()
    print("⚠️ No trajectory data available for failure detection")

## 8. Failure Rate Visualization and Reusability Summary (Protocol Requirement)

Visualize failure rates and sensor reusability to assess compliance with industry standards.


In [ ]:
# Failure Rate Visualization and Reusability Summary
# --------------------------------------------------------------------------------------------------


def plot_failure_rate_summary(failures_df: pd.DataFrame) -> None:
    """Plot failure rate summary."""
    if failures_df.empty:
        print("⚠️ No failure data to plot")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Failure status bar chart
    ax1 = axes[0]
    failure_counts = failures_df["failed"].value_counts()
    labels = ["Operational", "Failed"]
    colors = ["green", "red"]
    counts = [failure_counts.get(False, 0), failure_counts.get(True, 0)]

    bars = ax1.bar(labels, counts, color=colors, alpha=0.7)
    ax1.set_ylabel("Number of Sensors")
    ax1.set_title("Sensor Failure Status")
    ax1.grid(True, axis="y", alpha=0.3)

    # Add count labels on bars
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax1.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{count}\n({count / len(failures_df) * 100:.1f}%)",
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )

    # Plot 2: Uses before failure histogram
    ax2 = axes[1]

    if failures_df["failed"].sum() > 0:
        failed_sensors = failures_df[failures_df["failed"]]
        ax2.hist(
            failed_sensors["uses_before_failure"],
            bins=10,
            color="red",
            alpha=0.7,
            edgecolor="black",
        )
        ax2.axvline(
            failed_sensors["uses_before_failure"].mean(),
            color="darkred",
            linestyle="--",
            linewidth=2,
            label=f"Mean: {failed_sensors['uses_before_failure'].mean():.1f}",
        )
        ax2.axvline(
            failed_sensors["uses_before_failure"].median(),
            color="orange",
            linestyle="--",
            linewidth=2,
            label=f"Median: {failed_sensors['uses_before_failure'].median():.1f}",
        )
        ax2.set_xlabel("Number of Uses Before Failure")
        ax2.set_ylabel("Number of Sensors")
        ax2.set_title("Distribution of Uses Before Failure\n(Failed Sensors Only)")
        ax2.legend()
    else:
        ax2.text(
            0.5,
            0.5,
            "No sensor failures detected",
            ha="center",
            va="center",
            transform=ax2.transAxes,
            fontsize=12,
        )
        ax2.set_title("Distribution of Uses Before Failure")

    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_reusability_summary(failures_df: pd.DataFrame) -> None:
    """Plot reusability summary showing total uses per sensor."""
    if failures_df.empty:
        print("⚠️ No failure data to plot")
        return

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: Total uses per sensor (bar chart)
    ax1 = axes[0]
    sensors = natural_sort(failures_df["sensor_id"].astype(str).tolist())
    uses = [failures_df[failures_df["sensor_id"] == s]["total_uses"].iloc[0] for s in sensors]
    colors = [
        "red" if failures_df[failures_df["sensor_id"] == s]["failed"].iloc[0] else "green"
        for s in sensors
    ]

    ax1.bar(range(len(sensors)), uses, color=colors, alpha=0.7)
    ax1.set_xticks(range(len(sensors)))
    ax1.set_xticklabels(sensors, rotation=45, ha="right")
    ax1.set_ylabel("Total Number of Uses")
    ax1.set_xlabel("Sensor ID")
    ax1.set_title("Sensor Reusability: Total Uses Per Sensor\n(Green=Operational, Red=Failed)")
    ax1.grid(True, axis="y", alpha=0.3)

    # Plot 2: Uses before failure vs operational uses (boxplot)
    ax2 = axes[1]

    failed_uses = (
        failures_df[failures_df["failed"]]["uses_before_failure"].tolist()
        if failures_df["failed"].sum() > 0
        else []
    )
    operational_uses = (
        failures_df[~failures_df["failed"]]["total_uses"].tolist()
        if (~failures_df["failed"]).sum() > 0
        else []
    )

    data_to_plot = []
    labels_to_plot = []

    if failed_uses:
        data_to_plot.append(failed_uses)
        labels_to_plot.append("Failed Sensors\n(Uses Before Failure)")
    if operational_uses:
        data_to_plot.append(operational_uses)
        labels_to_plot.append("Operational Sensors\n(Total Uses)")

    if data_to_plot:
        bp = ax2.boxplot(data_to_plot, labels=labels_to_plot, patch_artist=True)
        colors_box = ["red" if "Failed" in label else "green" for label in labels_to_plot]
        for patch, color in zip(bp["boxes"], colors_box):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax2.set_ylabel("Number of Uses")
        ax2.set_title("Reusability Comparison: Failed vs Operational Sensors")
        ax2.grid(True, axis="y", alpha=0.3)
    else:
        ax2.text(
            0.5,
            0.5,
            "Insufficient data",
            ha="center",
            va="center",
            transform=ax2.transAxes,
            fontsize=12,
        )

    plt.tight_layout()
    plt.show()


# Generate visualizations
if not failures_df.empty:
    plot_failure_rate_summary(failures_df)
    plot_reusability_summary(failures_df)

    # Print reusability assessment
    print("\n" + "=" * 80)
    print("REUSABILITY ASSESSMENT (Protocol Requirement)")
    print("=" * 80)

    n_total = len(failures_df)
    n_failed = failures_df["failed"].sum()
    n_operational = n_total - n_failed

    print("\n📊 Overall Statistics:")
    print(f"  Total sensors tested: {n_total}")
    print(f"  Sensors that failed: {n_failed} ({n_failed / n_total * 100:.1f}%)")
    print(f"  Sensors still operational: {n_operational} ({n_operational / n_total * 100:.1f}%)")

    if n_failed > 0:
        failed_uses = failures_df[failures_df["failed"]]["uses_before_failure"]
        print("\n🔴 Failed Sensors:")
        print(f"  Mean uses before failure: {failed_uses.mean():.1f}")
        print(f"  Median uses before failure: {failed_uses.median():.1f}")
        print(f"  Range: {failed_uses.min():.0f} - {failed_uses.max():.0f} uses")

    if n_operational > 0:
        operational_uses = failures_df[~failures_df["failed"]]["total_uses"]
        print("\n🟢 Operational Sensors:")
        print(f"  Mean total uses: {operational_uses.mean():.1f}")
        print(f"  Median total uses: {operational_uses.median():.1f}")
        print(f"  Range: {operational_uses.min():.0f} - {operational_uses.max():.0f} uses")
        print(f"  Minimum reusability (n): {operational_uses.min():.0f} uses")

    # Protocol requirement: "Each sensor will maintain performance without signal degradation for n consecutive uses"
    if n_operational > 0:
        min_reusability = operational_uses.min()
        print("\n✅ Protocol Assessment:")
        print(f"  Minimum reusability (n): {min_reusability:.0f} consecutive uses")
        print(
            f"  {n_operational}/{n_total} sensors ({n_operational / n_total * 100:.1f}%) maintained performance"
        )
        print(f"  Failure rate: {n_failed / n_total * 100:.1f}%")
        print("\n  Note: Industry standard comparison requires specific thresholds.")
        print("        Current analysis uses statistical thresholds based on baseline performance.")
else:
    print("⚠️ No failure data available for visualization")